In [ ]:
import numpy as np
from numpy.linalg import norm
from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [ ]:
# Point the client to your local Ollama instance
client = OpenAI(
    base_url='https://ollama.ourhomelab.com/v1',
    api_key='ollama', # The key is ignored but required by the library
)

def get_llm_response(prompt):
    response = client.chat.completions.create(
        model="gemma4:12b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
# Example usage:
print(get_llm_response("How do I bake a cake in 100 words?"))

### Lesson 1

In [ ]:
KNOWLEDGE_BASE = {
    "doc1": {
        "title": "Project Chimera Overview",
        "content": (
            "Project Chimera is a research initiative focused on developing "
            "novel bio-integrated interfaces. It aims to merge biological "
            "systems with advanced computing technologies."
        )
    },
    "doc2": {
        "title": "Chimera's Neural Interface",
        "content": (
            "The core component of Project Chimera is a neural interface "
            "that allows for bidirectional communication between the brain "
            "and external devices. This interface uses biocompatible "
            "nanomaterials."
        )
    },
    "doc3": {
        "title": "Applications of Chimera",
        "content": (
            "Potential applications of Project Chimera include advanced "
            "prosthetics, treatment of neurological disorders, and enhanced "
            "human-computer interaction. Ethical considerations are paramount."
        )
    }
}

In [ ]:
def rag_retrieval(query, documents):
    query_words = set(query.lower().split())
    best_doc_id = None
    best_overlap = 0

    for doc_id, doc in documents.items():
        #Compare the query_words with document's content words 
        doc_words = set(doc["content"].lower().split()).union(set(doc["title"].lower().split()))
        overlap = len(query_words.intersection(doc_words))
        if overlap > best_overlap:
            best_overlap = overlap
            best_doc_id = doc_id

    return documents.get(best_doc_id)

In [ ]:
def rag_retrieval_multi(query, documents):
    query_words = set(query.lower().split())
    matched_docs = []

    for doc_id, doc in documents.items():
        #Compare the query_words with document's content words 
        doc_words = set(doc["content"].lower().split()).union(set(doc["title"].lower().split()))
        overlap = len(query_words.intersection(doc_words))
        if overlap:
            matched_docs.append(doc)

    return matched_docs

In [ ]:
def rag_generation(query, document):
    """
    This function augments the user's original question with the retrieved document's content. 
    """
    if document:
        snippet = f"{document['title']}: {document['content']}"
        prompt = f"Using the following information: `{snippet}`, answer the question: {query}"
    else:
        prompt = f"No relevant information found. Answer the question directly: {query}"
    return get_llm_response(prompt)

In [ ]:
def rag_generation_multi(query, documents):
    """
    This function augments the user's original question with the retrieved document's content. 
    """
    if documents:
        snippet = ''
        for document in documents:
            snippet += f"{document['title']}: {document['content']}; "
        prompt = f"Using the following information: `{snippet}`, answer the question: {query}"
    else:
        prompt = f"No relevant information found. Answer the question directly: {query}"
    return get_llm_response(prompt)

In [ ]:
def naive_generation(query):
    # This approach ignores the knowledge base
    prompt = f"Answer directly the following question: {query}"
    return get_llm_response(prompt)

In [ ]:
# query = "What is the main goal of Project Chimera?"
query = "What are the applications of Project Chimera?"

In [ ]:
naive_answer = naive_generation(query)
print("Naive approach:", naive_answer)

In [ ]:
doc = rag_retrieval(query, KNOWLEDGE_BASE)
rag_answer = rag_generation(query, doc)
print("RAG approach:", rag_answer)

In [ ]:
doc = rag_retrieval_multi(query, KNOWLEDGE_BASE)
rag_answer = rag_generation_multi(query, doc)
print("RAG approach:", rag_answer)

In [ ]:
KNOWLEDGE_BASE = {
    "AAPL": {
        "title": "AAPL Stock (April 2023)",
        "content": (
            "On 2023-04-13, AAPL opened at $160.50, closed at $162.30, with a high of $163.00 and a low of $159.90. "
            "Trading volume was 80 million shares. "
            "On 2023-04-14, AAPL opened at $161.10, closed at $162.80, with a high of $163.50 and a low of $160.50. "
            "Trading volume was 85 million shares."
        )
    },
    "MSFT": {
        "title": "MSFT Stock (April 2023)",
        "content": (
            "On 2023-04-13, MSFT opened at $285.00, closed at $288.50, with a high of $290.00 and a low of $283.50. "
            "Trading volume was 35 million shares. "
            "On 2023-04-14, MSFT opened at $286.00, closed at $289.00, with a high of $291.50 and a low of $284.70. "
            "Trading volume was 40 million shares."
        )
    },
    "TSLA": {
        "title": "TSLA Stock (April 2023)",
        "content": (
            "On 2023-04-13, TSLA opened at $185.00, closed at $187.00, with a high of $189.00 and a low of $184.50. "
            "Trading volume was 50 million shares. "
            "On 2023-04-14, TSLA opened at $186.00, closed at $188.50, with a high of $190.00 and a low of $185.50. "
            "Trading volume was 55 million shares."
        )
    }
}

In [ ]:
query = (
    "Write a short summary of the stock market performance on April 14, "
    "2023 for the following symbols: AAPL, MSFT, TSLA.\n"
    "Your summary should include:\n"
    "For each symbol:\n"
    "- The opening price\n"
    "- The closing price\n"
    "- The highest and lowest prices of the day\n"
    "- The trading volume"
)

In [ ]:
naive_answer = naive_generation(query)
print("Naive approach:", naive_answer)

In [ ]:
doc = rag_retrieval_multi(query, KNOWLEDGE_BASE)
rag_answer = rag_generation_multi(query, doc)
print("RAG approach:", rag_answer)

### Lesson 2

In [ ]:
def build_vocab(docs):
    unique_words = set()
    for doc in docs:
        for word in doc.lower().split():
            clean_word = word.strip(".,!?")
            if clean_word:
                unique_words.add(clean_word)
    return {word: idx for idx, word in enumerate(sorted(unique_words))}


def bow_vectorize(text, vocab):
    # TODO: Create a zero vector with the same length as the vocabulary
    bow_vector = np.zeros(len(vocab), dtype=int)
    # TODO: Convert the text to lowercase and split it into words
    tokens = text.lower().split()
    # TODO: For each word, clean it by stripping punctuation
    for word in tokens:
        cleaned_word = word.strip(".,!?")
    # TODO: Check if the cleaned word is in the vocabulary
        if cleaned_word in vocab:
    # TODO: If the word is in the vocabulary, increment the corresponding index in the vector
            bow_vector[vocab[cleaned_word]] += 1
    return bow_vector


if __name__ == "__main__":
    example_texts = [
        "RAG stands for retrieval augmented generation, and retrieval is a key component of RAG.",
        "Data is crucial for retrieval processes, and without data, retrieval systems cannot function effectively."
    ]

    vocab = build_vocab(example_texts)
    print("Vocabulary: ", vocab.items(), '\n')

    for txt in example_texts:
        vec = bow_vectorize(txt, vocab)
        print(f"Text: {txt}\nBOW Vector: {vec}\n")

In [ ]:
def bow_search(query, docs):
    """
    Rank documents by lexical overlap using the BOW technique. 
    The dot product between the query vector and each document vector 
    indicates how many words they share.
    """
    query_vec = bow_vectorize(query, VOCAB)
    scores = []
    for i, doc in enumerate(docs):
        doc_vec = bow_vectorize(doc, VOCAB)
        score = np.dot(query_vec, doc_vec)  # Higher score = more overlap
        scores.append((i, score))
    # Sort by descending overlap
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

In [ ]:
t = ["I love machine learning", "Machine learning is fun"]

In [ ]:
build_vocab(t)

In [ ]:
" ".join(t)

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
sentences = [
    "RAG stands for Retrieval Augmented Generation.",
    "A Large Language Model is a Generative AI model for text generation.",
    "RAG enhance text generation of LLMs by incorporating external data",
    "Bananas are yellow fruits.",
    "Apples are good for your health.",
    "What's monkey's favorite food?"
]

embeddings = model.encode(sentences, show_progress_bar=True)

In [ ]:
print(embeddings.shape)  # e.g., (6, 384), depending on the model
print(embeddings[0])     # A sample embedding for the first sentence

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """
    Compute cosine similarity between two vectors.
    Range: -1 (opposite directions) to 1 (same direction).
    """
    return np.dot(vec_a, vec_b) / (norm(vec_a) * norm(vec_b))

In [ ]:
for i, sent_i in enumerate(sentences):
    # Start the inner loop from the next sentence to avoid redundant comparisons
    for j, sent_j in enumerate(sentences[i+1:], start=i+1):
        sim_score = cosine_similarity(embeddings[i], embeddings[j])
        print(f"Similarity('{sent_i}' , '{sent_j}') = {sim_score:.4f}")

In [ ]:
cosine_similarity(embeddings[0], embeddings[1])

In [ ]:
for i, sent_i in enumerate(embeddings):
    for j, sent_j in enumerate(embeddings[i+1:], start = i+1):
        # print(i, j)
        print(cosine_similarity(sent_i, sent_j))

In [ ]:
def cos_sim(a, b):
    """
    Compute cosine similarity between two vectors, 
    indicating how similar they are.
    """
    return np.dot(a, b) / (norm(a) * norm(b))

def embedding_search(query, docs, model):
    """
    Rank documents by comparing how semantically close they are 
    to the query in the embedding space using cosine similarity.
    """
    # Encode both the query and documents into embeddings
    query_emb = model.encode([query])[0]
    doc_embs = model.encode(docs)

    scores = []
    for i, emb in enumerate(doc_embs):
        score = cos_sim(query_emb, emb)
        scores.append((i, score))
    # Sort by semantic similarity in descending order
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

In [ ]:
def get_sentences_and_categories():
    """
    Return the sentences and their corresponding categories.
    """
    sentences = [
        # Topic: NLP
        "RAG stands for Retrieval-Augmented Generation.",
        "Retrieval is a crucial aspect of modern NLP systems.",
        "Generating text with correct facts is challenging.",
        "Large language models can generate coherent text.",
        "GPT models have billions of parameters.",
        "Natural Language Processing enables computers to understand human language.",
        "Word embeddings capture semantic relationships between words.",
        "Transformer architectures revolutionized NLP research.",
        
        # Topic: Machine Learning
        "Machine learning benefits from large datasets.",
        "Supervised learning requires labeled data.",
        "Reinforcement learning is inspired by behavioral psychology.",
        "Neural networks can learn complex functions.",
        "Overfitting is a common problem in ML.",
        "Unsupervised learning uncovers hidden patterns in data.",
        "Feature engineering is critical for model performance.",
        "Cross-validation helps in assessing model generalization.",
        
        # Topic: Food
        "Bananas are commonly used in smoothies.",
        "Oranges are rich in vitamin C.",
        "Pizza is a popular Italian dish.",
        "Cooking pasta requires boiling water.",
        "Chocolate can be sweet or bitter.",
        "Fresh salads are a healthy and refreshing meal.",
        "Sushi combines rice, fish, and seaweed in a delicate balance.",
        "Spices can transform simple ingredients into gourmet dishes.",
        
        # Topic: Weather
        "It often rains in the Amazon rainforest.",
        "Summers can be very hot in the desert.",
        "Hurricanes form over warm ocean waters.",
        "Snowstorms can disrupt transportation.",
        "A sunny day can lift people's mood.",
        "Foggy mornings are common in coastal regions.",
        "Winter brings frosty nights and chilly winds.",
        "Thunderstorms can produce lightning and heavy rain."
    ]
    
    categories = (["NLP"] * 8 + ["ML"] * 8 + ["Food"] * 8 + ["Weather"] * 8)
    return sentences, categories

def get_color_and_shape_maps():
    """
    Return color and marker maps for each category.
    """
    color_map = {
        "NLP": "red",
        "ML": "blue",
        "Food": "green",
        "Weather": "purple"
    }
    shape_map = {
        "NLP": "o",
        "ML": "s",
        "Food": "^",
        "Weather": "X"
    }
    return color_map, shape_map

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

def compute_tsne_embeddings(sentences, model_name="sentence-transformers/all-MiniLM-L6-v2",
                            perplexity=10, n_iter=3000, random_state=42):
    """
    Compute and return t-SNE reduced embeddings for the given sentences.
    """
    # 1. Initialize a SentenceTransformer model that balances speed and performance.
    model = SentenceTransformer(model_name)
    
    # 2. Convert each sentence into a high-dimensional embedding.
    embeddings = model.encode(sentences)
    
    # 3. Configure t-SNE with chosen parameters and reduce embeddings to 2D.
    tsne = TSNE(n_components=2, random_state=random_state,
                perplexity=perplexity, n_iter=n_iter)
    
    # 4. Fit t-SNE on the embeddings and return a 2D representation.
    return tsne.fit_transform(embeddings)

def plot_embeddings(reduced_embeddings, sentences, categories, color_map, shape_map,
                    xlim=(-125, 150), ylim=(-175, 125)):
    """
    Plot the 2D embeddings with labels and a legend.
    """
    # 1. Create a figure to hold the scatter plot.
    plt.figure(figsize=(10, 8))
    
    # 2. Plot each sentence:
    #    - Use the category to decide color and marker shape.
    #    - Use the first 20 characters as a short text label.
    for i, (sentence, category) in enumerate(zip(sentences, categories)):
        x, y = reduced_embeddings[i]
        plt.scatter(x, y, color=color_map[category], marker=shape_map[category])
        plt.text(x - 2.5, y - 7.5, sentence[:20] + "...", fontsize=9)
    
    # 3. Construct a legend by plotting empty points, one for each category.
    for cat, color in color_map.items():
        plt.scatter([], [], color=color, label=cat, marker=shape_map[cat])
    plt.legend(loc="best")

    # 4. Add labels, set boundaries, and save the final plot.
    plt.title("t-SNE Visualization of Sentence Embeddings", fontsize=14)
    plt.xlabel("t-SNE Dimension 1", fontsize=12)
    plt.ylabel("t-SNE Dimension 2", fontsize=12)
    plt.tight_layout()
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.savefig('your_plot_image.png')  # Saving to an image file of your choice.    

### Lesson 3

In [ ]:
import os
import json
import re

def chunk_text(text, chunk_size=30):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of chunk strings.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    if text:
        chunks = re.split("[.?!]+", text)
    else:
        return []
    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed

    chunk_collector = []
    word_count = 0
    current_chunk = []
    
    for chunk in chunks:
        chunk_tokens = chunk.split()
        chunk_length = len(chunk_tokens)
        if word_count + chunk_length > chunk_size:
            chunk_collector.append(" ".join(current_chunk))
            word_count = 0
            current_chunk = []
        current_chunk.extend(chunk_tokens)
        word_count += chunk_length
    
    if current_chunk:
        chunk_collector.append(" ".join(current_chunk))
    
    return chunk_collector
            
    # # For now, this is the basic word-count-based implementation
    # words = text.split()
    # return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def load_and_chunk_dataset(file_path, chunk_size=30):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)
    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        doc_category = doc.get("category", "general")
        doc_chunks = chunk_text(doc_text, chunk_size)
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "category": doc_category,
                "text": chunk_str
            })
    return all_chunks

if __name__ == "__main__":
    current_dir = os.path.dirname(__file__)
    dataset_file = os.path.join(current_dir, "data", "corpus.json")
    chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=30)
    print("Loaded and chunked", len(chunked_docs), "chunks from dataset.")
    for c in chunked_docs:
        print(c)

In [ ]:
from chromadb import Client

In [ ]:
import os
import json
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions


def chunk_text(text, chunk_size=50):
    """
    Splits the given text into chunks of size 'chunk_size'.
    Returns a list of chunk strings.
    """
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]


def load_and_chunk_dataset(file_path, chunk_size=50):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    # TODO: Open and load the JSON file
    with open(file_path) as f:
        data = f.read()
    
    all_chunks = []
    for doc in data:
        doc_text = doc["content"]
        # TODO: Extract category and id from the document
        doc_category = doc["category"]
        doc_id = doc["id"]
        
        doc_chunks = chunk_text(doc_text, chunk_size)
        for chunk_index, chunk_str in enumerate(doc_chunks):
            # TODO: Create a dictionary for each chunk with doc_id, chunk_id, category and text
            d = {}
            d["doc_id"] = doc_id
            d["chunk_id"] = chunk_index
            d["category"] = doc_category
            d["text"] = chunk_str
            
    return all_chunks


def build_chroma_collection(chunks, collection_name="rag_collection"):
    """
    Builds or retrieves a ChromaDB collection, embedding each chunk using a SentenceTransformer.
    Adds all chunks in the 'chunks' list to the collection for fast retrieval.
    """
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

    client = Client(Settings())
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embed_func
    )

    texts = [c["text"] for c in chunks]
    ids = [f"chunk_{c['doc_id']}_{c['chunk_id']}" for c in chunks]
    metadatas = [
        {"doc_id": c["doc_id"], "chunk_id": c["chunk_id"], "category": c["category"]}
        for c in chunks
    ]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    return collection


if __name__ == "__main__":
    current_dir = os.path.dirname(__file__)
    dataset_file = os.path.join(current_dir, "data", "corpus.json")

    chunked_docs = load_and_chunk_dataset(dataset_file)
    collection = build_chroma_collection(chunked_docs)

    total_docs = collection.count()
    print("ChromaDB collection created with", total_docs, "documents.")

In [ ]:
import json
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from scripts.llm import get_llm_response


def retrieve_top_chunks(query, collection, top_k=3):
    """
    Retrieves the top_k chunks most relevant to the given query from 'collection'.
    Returns a list of retrieved chunks, each containing 'chunk' text, 'doc_id', and 'distance'.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    retrieved_chunks = []
    
    # Safeguard if no documents are returned
    if not results['documents'][0]:
        return retrieved_chunks

    for i in range(len(results['documents'][0])):
        retrieved_chunks.append({
            "chunk": results['documents'][0][i],
            "doc_id": results['ids'][0][i],
            "distance": results['distances'][0][i]
        })
    return retrieved_chunks


def build_prompt(query, retrieved_chunks):
    """
    Constructs an LLM prompt by combining multiple retrieved chunks into a
    single context block, ensuring the model can handle longer or more detailed answers.
    """
    # TODO: Implement the build_prompt function that creates a prompt string
    # combining the query and retrieved chunks
    prompt = f"Question: {query} \n"
    if retrieved_chunks:
        prompt += f"Answer the question using the following context: \n"
        for chunk in retrieved_chunks:
            prompt += f"- {chunk['chunk']} \n"
    else:
        prompt += "Answer: "
    return prompt

if __name__ == "__main__":
    # Load corpus data from JSON file
    with open('data/corpus.json', 'r') as f:
        corpus_data = json.load(f)

    # Set up the embedding model and initialize a ChromaDB collection
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
    client = Client(Settings())
    collection = client.get_or_create_collection("rag_collection", embedding_function=embed_func)

    # Add documents from corpus_data into the vector database
    documents = [doc['content'] for doc in corpus_data]
    ids = [f"chunk_{doc['id']}_0" for doc in corpus_data]
    collection.add(documents=documents, ids=ids)

    # Define a sample query
    query = "What are some recent technological breakthroughs?"

    # TODO: Retrieve chunks, build the prompt, and get the LLM response
    retrieved_chunks = retrieve_top_chunks(query, collection, top_k=5)
    prompt = build_prompt(query, retrieved_chunks)
    
    # TODO: Print the final prompt and LLM answer
    print("Prompt: ", prompt)
    print("LLM Answer: ", get_llm_response(prompt))

In [ ]:
import json
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from scripts.llm import get_llm_response


def retrieve_top_chunks(query, collection, category=None, top_k=3):
    """
    Retrieves the top_k chunks most relevant to the given query from 'collection',
    optionally filtered by category. Returns a list of retrieved chunks, each
    containing 'chunk' text, 'doc_id', and 'distance'.
    """
    # TODO: Create a where dictionary to filter by category if one is provided
    where = {"category": category}

    # TODO: Perform the query with metadata filtering using collection.query()
    # Include the where parameter in the query
    results = collection.query(query_texts=[query], where=where, n_results=top_k)

    retrieved_chunks = []

    # Safeguard against empty results
    if not results['documents'] or not results['documents'][0]:
        return retrieved_chunks

    # TODO: Process query results and append each chunk's information to retrieved_chunks
    for i in range(len(results["documents"][0])):
        chunk_info = {}
        chunk_info["chunk"] = results["documents"][0][i]
        chunk_info["doc_id"] = results["ids"][0][i]
        chunk_info["distance"] = results["distances"][0][i]
        retrieved_chunks.append(chunk_info)
    return retrieved_chunks


def build_prompt(query, retrieved_chunks):
    """
    Constructs an LLM prompt by combining multiple retrieved chunks into a
    single context block, ensuring the model can provide context-based answers.
    """
    prompt = f"Question: {query}\nAnswer using only the following context:\n"
    for rc in retrieved_chunks:
        prompt += f"- {rc['chunk']}\n"
    prompt += "Answer:"
    return prompt


if __name__ == "__main__":
    # Load corpus data from JSON file
    with open('data/corpus.json', 'r') as f:
        corpus_data = json.load(f)

    # Prepare documents, ids, and metadatas
    documents = [doc['content'] for doc in corpus_data]
    ids = [f"chunk_{doc['id']}_0" for doc in corpus_data]
    metadatas = [{"category": doc["category"]} for doc in corpus_data]

    # Create or retrieve the vector database collection
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
    client = Client(Settings())
    collection = client.get_or_create_collection(
        "rag_collection",
        embedding_function=embed_func
    )

    # Add documents with metadata to the collection
    collection.add(documents=documents, ids=ids, metadatas=metadatas)

    # TODO: Define a query and category to test the retrieval function
    user_query = "What are some recent technological breakthroughs?"
    user_category = "Technology"

    # TODO: Retrieve chunks matching the query and category
    retrieved = retrieve_top_chunks(user_query, collection, user_category)

    # TODO: Implement logic to handle empty results, build prompt, and get LLM response
    # Print appropriate messages or the final prompt and answer
    if not retrieved:
        print("No matching context found.")
        prompt = "Answer the following question: " + user_query
        llm_response = get_llm_response(prompt)
    else:
        prompt = build_prompt(user_query, retrieved)
        llm_response = get_llm_response(prompt)
    print("Prompt: ", prompt)
    print("LLM Response", llm_response)

In [ ]:
import json
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions


def metadata_enhanced_search(query, collection, categories=None, top_k=3):
    """
    Takes a query, a ChromaDB collection, an optional list of categories, and
    returns the top_k most relevant documents. If categories is specified, only
    documents matching any of those categories are retrieved.
    """
    # Build the metadata filter if categories were specified
    where_clause = {"category": {"$in": categories}} if categories else None

    # TODO: Use collection.query to search for relevant documents.
    # Remember to pass the query, number of results, and the where clause.
    results = collection.query(query_texts=[query], where=where_clause, n_results=top_k)

    # TODO: Process the results and create a list of dictionaries.
    # Each dictionary should contain the chunk content, document ID, category, and distance.
    retrieved_chunks = []
    for i in range(len(results["documents"][0])):
        chunk_info = {}
        chunk_info["chunk"] = results["documents"][0][i]
        chunk_info["doc_id"] = results['metadatas'][0][i]['doc_id']
        chunk_info["distance"] = results["distances"][0][i]
        chunk_info["category"] = results["metadatas"][0][i].get('category')
        retrieved_chunks.append(chunk_info)    

    return retrieved_chunks


if __name__ == "__main__":
    # Load sample data from JSON file
    with open("data/corpus.json", "r") as f:
        sample_chunks = json.load(f)

    # Create a ChromaDB client and set up the embedding function
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
    client = Client(Settings())

    # Create or get a collection for our metadata demo
    collection = client.create_collection(
        "metadata_demo_collection",
        embedding_function=embed_func
    )

    # Remove existing data (if any) and add fresh documents
    existing_ids = collection.get().get("ids", [])
    if existing_ids:
        collection.delete(ids=existing_ids)

    texts = [doc["content"] for doc in sample_chunks]
    doc_ids = [f"doc_{doc['id']}" for doc in sample_chunks]
    metadatas = [{
        "doc_id": doc["id"],
        "category": doc.get("category", "General"),
        "title": doc["title"],
        "date": doc["date"]
    } for doc in sample_chunks]

    # Add documents with their metadata to the ChromaDB collection
    collection.add(documents=texts, metadatas=metadatas, ids=doc_ids)

    # Define a query to demonstrate searching with and without metadata filtering
    query_input = "Recent advancements in AI and their impact on teaching"

    # Search WITHOUT category filtering
    print("======== WITHOUT CATEGORY FILTER ========")
    no_filter_results = metadata_enhanced_search(query_input, collection, categories=None, top_k=3)
    for res in no_filter_results:
        print(f"Doc ID: {res['doc_id']}, Category: {res['category']}, Distance: {res['distance']:.4f}")
        print(f"Chunk: {res['chunk']}\n")

    # Search WITH category filtering
    filter_category = "Education"
    print(f"======== WITH CATEGORY FILTER ({filter_category}) ========")
    filter_results = metadata_enhanced_search(query_input, collection, categories=[filter_category], top_k=3)
    for res in filter_results:
        print(f"Doc ID: {res['doc_id']}, Category: {res['category']}, Distance: {res['distance']:.4f}")
        print(f"Chunk: {res['chunk']}\n")

In [ ]:
import json
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from datetime import datetime


def metadata_enhanced_search(query, collection, categories=None, min_date=None, top_k=3):
    """
    Filters documents by category and a minimum publication date.
    If a list of categories is provided, only documents in those categories are returned.
    If a min_date is provided, only documents with date >= min_date are returned.
    Both filters are combined such that documents must satisfy all provided conditions.
    """

    # TODO: Convert min_date to a timestamp if provided
    min_date_formatted = None
    if min_date: min_date_formatted = datetime.timestamp(datetime.fromisoformat(min_date))

    # TODO: Build a compound where clause that combines category and date filtering.
    # If both filters are provided, documents must match both conditions.
    # If only one filter is provided, use that one.
    # If no filters are provided, where_clause should be None.
    if categories is not None and min_date is not None:
        where_clause = {"$and":[{"category": {"$in": categories}} , {"date": {"$gte": min_date_formatted}}]}
    elif categories is not None and min_date is None:
        where_clause = {"category": {"$in": categories}}
    elif categories is None and min_date is not None:
        where_clause = {"date": {"$gte": min_date_formatted}}  
    else:
        where_clause = None      

    # Execute the query using the ChromaDB collection
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        where=where_clause
    )

    # Compile the retrieved documents into a list.
    retrieved_chunks = []
    for i in range(len(results['documents'][0])):
        retrieved_chunks.append({
            "chunk": results['documents'][0][i],
            "doc_id": results['metadatas'][0][i]['doc_id'],
            "category": results['metadatas'][0][i].get('category', "General"),
            "distance": results['distances'][0][i],
            "date": datetime.fromtimestamp(results['metadatas'][0][i].get('date', 0)).isoformat()
        })

    return retrieved_chunks


if __name__ == "__main__":
    # Load sample data from JSON file
    with open("data/corpus.json", "r") as f:
        sample_chunks = json.load(f)

    # Create a ChromaDB client and embedder
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
    client = Client(Settings())

    # Create or retrieve the collection
    collection = client.create_collection(
        "metadata_demo_collection",
        embedding_function=embed_func
    )

    # Clear any existing documents in the collection
    existing_ids = collection.get().get("ids", [])
    if existing_ids:
        collection.delete(ids=existing_ids)

    # Prepare data and add to the collection
    texts = [doc["content"] for doc in sample_chunks]
    doc_ids = [f"doc_{doc['id']}" for doc in sample_chunks]
    metadatas = []
    for doc in sample_chunks:
        # TODO: Convert date to timestamp and store it
        metadatas.append({
            "doc_id": doc["id"],
            "category": doc.get("category", "General"),
            "title": doc["title"],
            "date": datetime.timestamp(datetime.fromisoformat(doc["date"]))  # Store date as a timestamp
        })

    collection.add(documents=texts, metadatas=metadatas, ids=doc_ids)

    # Demonstrate searches
    query_input = "Recent advancements in AI and their impact on teaching"

    print("======== WITHOUT ANY FILTER ========")
    no_filter_results = metadata_enhanced_search(query_input, collection, categories=None, min_date=None, top_k=3)
    for res in no_filter_results:
        print(f"Doc ID: {res['doc_id']} | Category: {res['category']} | Date: {res['date']} | Distance: {res['distance']:.4f}")
        print(f"Chunk: {res['chunk']}\n")

    print("======== WITH CATEGORY FILTER (Education) ONLY ========")
    cat_only_results = metadata_enhanced_search(query_input, collection, categories=["Education"], min_date=None, top_k=3)
    for res in cat_only_results:
        print(f"Doc ID: {res['doc_id']} | Category: {res['category']} | Date: {res['date']} | Distance: {res['distance']:.4f}")
        print(f"Chunk: {res['chunk']}\n")

    print("======== WITH CATEGORY FILTER (Education) AND DATE FILTER (>= 2022-01-01) ========")
    cat_and_date_results = metadata_enhanced_search(query_input, collection, categories=["Education"], min_date="2022-01-01", top_k=3)
    for res in cat_and_date_results:
        print(f"Doc ID: {res['doc_id']} | Category: {res['category']} | Date: {res['date']} | Distance: {res['distance']:.4f}")
        print(f"Chunk: {res['chunk']}\n")

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from data import load_and_chunk_corpus
from vector_db import build_chroma_collection

def build_bm25_index(chunks):
    """
    Build a BM25Okapi index from the chunk texts for lexical-based retrieval.
    Reimplemented from scratch by lowercasing and splitting each chunk into tokens.
    """
    # TODO: Initialize an empty list to store tokenized chunks
    corpus = []

    # TODO: Tokenize each chunk's text by converting to lowercase and splitting into words
    for chunk in chunks:
        corpus.append(chunk["text"].lower().split())

    # TODO: Create and return the BM25Okapi index using the tokenized corpus
    return BM25Okapi(corpus)

def hybrid_retrieval(query, chunks, bm25, collection, top_k=3, alpha=0.5):
    """
    Merge BM25 and embedding-based results.
    Steps:
      1) Compute BM25 scores for each chunk. (Higher = better)
      2) Get embedding distances from ChromaDB for a candidate set.
      3) Convert distances to similarity (e.g., similarity ~ 1/(1+distance)).
      4) Normalize both BM25 and similarity to [0,1] and combine with weighting:
         final_score = alpha * BM25_normalized + (1-alpha) * embedding_similarity
      5) Sort by final score in descending order.

    'alpha' controls how much weight lexical vs. embedding-based similarity gets.
    In practice, you might do cross-validation or user acceptance testing to find a good alpha.
    """
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_min, bm25_max = (min(bm25_scores), max(bm25_scores)) if bm25_scores.size > 0 else (0, 1)

    embed_results = collection.query(query_texts=[query], n_results=min(top_k*5, len(chunks)))
    embed_scores_dict = {}
    for i in range(len(embed_results['documents'][0])):
        idx = embed_results['ids'][0][i]
        distance = embed_results['distances'][0][i]
        similarity = 1 / (1 + distance)
        embed_scores_dict[idx] = similarity

    merged = []
    for i, chunk in enumerate(chunks):
        bm25_raw = bm25_scores[i]
        if bm25_max != bm25_min:
            bm25_norm = (bm25_raw - bm25_min) / (bm25_max - bm25_min)
        else:
            bm25_norm = 0.0

        embed_sim = embed_scores_dict.get(i, 0.0)
        final_score = alpha * bm25_norm + (1 - alpha) * embed_sim
        merged.append((i, final_score))

    merged.sort(key=lambda x: x[1], reverse=True)
    top_results = merged[:top_k]

    print(f"Top results by combined BM25 + embeddings for query: '{query}'")
    for idx, score in top_results:
        print(f"Chunk: '{chunks[idx]['text'][:50]}...' | Score: {score:.4f}")
    return [(idx, chunks[idx], score) for (idx, score) in top_results]


if __name__ == "__main__":
    chunked_docs = load_and_chunk_corpus("data/corpus.json", 40)
    bm25_index = build_bm25_index(chunked_docs)
    collection = build_chroma_collection(chunked_docs, collection_name="hybrid_collection")

    query = "What do our internal company policies state?"
    results = hybrid_retrieval(query, chunked_docs, bm25_index, collection, top_k=3, alpha=0.6)
    if not results:
        print("No chunks found. Fallback to a naive or apology answer.")
    else:
        for chunk_idx, chunk_data, final_score in results:
            print(f"Chunk ID: {chunk_idx}, Score: {final_score:.4f}, Text: {chunk_data['text']}")